In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import (confusion_matrix,accuracy_score,
recall_score,f1_score,roc_auc_score,precision_score)

In [34]:
df=pd.read_csv(r'C:\Users\USER\Desktop\5th sem\project1_daibatise\data\processed\mohammed_ajlan_p1w1_features.csv')

In [35]:
feature_cols = ["age","pregnancies","glucose","blood_pressure",
                "skin_thickness","insulin","bmi","diabetes_pedigree",
                "bmi_category","glucose_band"]
x = df[feature_cols].copy()
y = df["outcome"]
x

,age,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,bmi_category,glucose_band
0,79,0,124.000000,71.0,30.0,124.0,23.4,0.588,normal,prediabetic
1,37,0,153.000000,85.0,24.0,42.0,33.9,0.192,obese,diabetic
2,39,0,142.000000,68.0,22.0,159.0,26.9,0.777,overweight,diabetic
3,68,7,121.000000,88.0,26.0,124.0,29.0,1.217,overweight,prediabetic
4,75,0,107.000000,84.0,31.0,101.0,21.2,0.278,normal,prediabetic
...,...,...,...,...,...,...,...,...,...,...
945,64,2,117.855165,81.0,34.0,124.0,40.3,0.641,obese,prediabetic
946,71,0,140.000000,64.0,37.0,124.0,33.4,0.116,obese,diabetic
947,40,0,130.000000,73.0,26.0,258.0,20.1,0.709,normal,diabetic
948,52,4,123.000000,76.0,22.0,189.0,38.9,0.540,obese,prediabetic


In [36]:
x=pd.get_dummies(x,drop_first=True)
x.shape

(950, 13)

In [37]:
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

In [38]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [39]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dtree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)



In [40]:
def score_row(name, pred,scores=None):
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    r = {
        "model":name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        "found": int(tp),
        "missed": int(fn),
        "false_alarms": int(fp)}
    if scores is  not None :
     r["auc"] = round(roc_auc_score(y_test,scores),4)
     return r
    print("Train:", X_train.shape, " Test:", X_test.shape)
print("Diabetic patients in the test set:", int(np.sum(y_test)), "of", len(y_test))

for nm, m, Xt in [
    ("Logistic Regression", logreg, x_test_scaled),
    ("Decision Tree (d=3)", dtree, x_test),
    ("KNN (k=25)", knn, x_test_scaled)
]:
    print(f"{nm:22s} accuracy {accuracy_score(y_test, m.predict(Xt)):.4f}")

Diabetic patients in the test set: 55 of 190
Logistic Regression    accuracy 0.7526
Decision Tree (d=3)    accuracy 0.7211
KNN (k=25)             accuracy 0.7474


restate the target you are aiming at


In [41]:
TARGET_RECALL = 0.70
GUARD_F1 = 0.50
BEST_SO_FAR = 0.6182

print("PROJECT METRIC : recall  (guard: F1 >=", GUARD_F1, ")")
print("TARGET          : recall >=", TARGET_RECALL)
print("BEST SO FAR     :", BEST_SO_FAR, "(Week 5 baseline at threshold 0.30)")
print("GAP TO CLOSE    :", round(TARGET_RECALL - BEST_SO_FAR, 4))

PROJECT METRIC : recall  (guard: F1 >= 0.5 )
TARGET          : recall >= 0.7
BEST SO FAR     : 0.6182 (Week 5 baseline at threshold 0.30)
GAP TO CLOSE    : 0.0818


SVM (Support Vector Machine) is a machine learning algorithm used mainly for classification.
It finds the best boundary (hyperplane) that separates different classes of data with the maximum possible margin between them.

train an svm with default settings

In [42]:
from sklearn.svm import SVC

svm_rbf = SVC(random_state=42)
svm_rbf.fit(x_train_scaled, y_train)

pred_rbf = svm_rbf.predict(x_test_scaled)

print("Fitted:", svm_rbf)
print("Patients flagged:", int(pred_rbf.sum()), "of", len(pred_rbf))

Fitted: SVC(random_state=42)
Patients flagged: 17 of 190


In [44]:
row_rbf= score_row("SVC rbf (default)",pred_rbf,
                   svm_rbf.decision_function(x_test_scaled))
print(pd.DataFrame([row_rbf]).to_string(index=False))
print()
print("target recall:", TARGET_RECALL,"| best so far:",BEST_SO_FAR)
print("this model :", row_rbf["recall"])

            model  accuracy  precision  recall     f1  found  missed  false_alarms    auc
SVC rbf (default)    0.7263     0.5882  0.1818 0.2778     10      45             7 0.6443

target recall: 0.7 | best so far: 0.6182
this model : 0.1818
